# Lab 13 - Multimodal Data Exploration

During this lab we are going to explore data of more than one modality - images and text.

## 1. Dataset

We will use ...

In [ ]:
# write your code here


## 2. Database

**This step is optional** - you can use any other framework or even manually manage the data.

To ease the data management and manipulation, we will use a database designed for machine learning
and AI applications, specialized for deep learning use cases - [Deep
Lake](https://github.com/activeloopai/deeplake) open-source database developed by Activeloop. They
provide richer environment and additional tools, however, for the purpose of this lab the basic
features of the database are enough.

Install the Deep Lake package and familiarize yourself with the API and documentation -
https://docs.deeplake.ai/latest. For example:
- create a local dataset
- add columns of different types: int, string, image, embedding
- add rows to the dataset

In [ ]:
# write your code here


## 3. Cross-modal search

Your task is to design a solution that allows users to search for images based on a text query, as
well as retrieve texts based on an image query. 

You should be able to query the system for a specified number of objects that are similar to a given
example item. The query item does not need to be present in the dataset. The system should support
queries in either of the two modalities:

- allowing users to enter a text query and retrieve the most similar images from the dataset,
- allowing users to provide an image query and retrieve the most similar texts from the dataset.

<span style="color:gold">Important note:</span> Make sure to implement the solution in a way that
you prepare aligned/joint/shared embeddings for the images and texts. For a text query, do not
perform text-to-text search followed by image retrieval. Instead, compute the embedding of the
input text and retrieve the most similar images directly based on embedding similarity. 

Consider using the following models to compute embeddings for the images and texts:

- SentenceTransformers
  (https://www.sbert.net/examples/sentence_transformer/applications/image-search/README.html)
- CLIP (https://huggingface.co/docs/transformers/model_doc/clip), e.g.,
  https://huggingface.co/openai/clip-vit-base-patch32
- Siglip (https://huggingface.co/docs/transformers/model_doc/siglip), e.g.,
  https://huggingface.co/google/siglip-base-patch16-224
- or any other model of your choice

Store the computed embeddings for images and texts and use them later to perform searches, for
example, using cosine similarity. You can either implement the search from scratch or use the Deep
Lake database query language with [vector
operations](https://docs.deeplake.ai/latest/advanced/tql/#vector-operations), or even introduce
[indexing](https://docs.deeplake.ai/latest/advanced/tql/#index-creation-for-optimal-performance) to
speed up the search.

In [ ]:
# write your code here

import deeplake
import numpy as np
from deeplake.types import Embedding, EmbeddingIndex, ClusteredQuantized

# Tworzymy lub otwieramy dataset
deeplake.delete('data/dataset2')
ds = deeplake.create('data/dataset2')

# Dodajemy embeddings i jakieś metadata
embeddings = np.random.rand(1000000, 24)  # przykładowe embeddingi

ds.add_column("pos", 'int32')
ds.add_column("embedding", Embedding(24, index_type=EmbeddingIndex(ClusteredQuantized)))

ds.append({"pos": np.arange(1000000), "embedding": embeddings})
ds.commit()

In [190]:
k = 100
text_vector = ','.join(str(x) for x in embeddings[k])
results = ds.query(f"""
    SELECT *
    ORDER BY COSINE_SIMILARITY(embedding, ARRAY[{text_vector}]) DESC
    LIMIT 3
""")
print(results[0]['pos'])
print(results[1]['pos'])
print(results[2]['pos'])

100
882567
100112


In [188]:
explanation = ds.explain_query(f"""
    SELECT * FROM "data/dataset2"
    WHERE COSINE_SIMILARITY(embedding, ARRAY[{text_vector}]) > 0.9
    LIMIT 3
""")
print(explanation)
explain_dict = explanation.to_dict()
print(f"Execution plan: {explain_dict}")

# Use explanation to optimize queries
if "index_used" in explain_dict:
    print("Query will use indexes for optimization")
else:
    print("Query will not use indexes for optimization")

Seq Scan on dataset
├── Filter: (COSINE_SIMILARITY(embedding, ARRAY[0.332268, 0.316658, 0.661787, 0.363573, 0.293287, 0.494711, 0.035018, 0.983019, 0.344806, 0.395443, 0.224208, 0.922944, 0.917482, 0.000156, 0.940689, 0.376213, 0.590671, 0.510595, 0.038578, 0.757781, 0.914312, 0.349699, 0.170851, 0.113930]) > 0.900000)
└── Limit (rows=3)
Execution plan: {'Plan': 'Seq Scan on dataset', 'children': [{'Plan': 'Filter: (COSINE_SIMILARITY(embedding, ARRAY[0.332268, 0.316658, 0.661787, 0.363573, 0.293287, 0.494711, 0.035018, 0.983019, 0.344806, 0.395443, 0.224208, 0.922944, 0.917482, 0.000156, 0.940689, 0.376213, 0.590671, 0.510595, 0.038578, 0.757781, 0.914312, 0.349699, 0.170851, 0.113930]) > 0.900000)'}, {'Plan': 'Limit (rows=3)'}]}
Query will not use indexes for optimization


In [189]:
explanation = ds.explain_query(f"""
    SELECT * FROM "data/dataset2"
    ORDER BY COSINE_SIMILARITY(embedding, ARRAY[{text_vector}]) DESC
    LIMIT 3
""")
print(explanation)
explain_dict = explanation.to_dict()
print(f"Execution plan: {explain_dict}")

# Use explanation to optimize queries
if "index_used" in explain_dict:
    print("Query will use indexes for optimization")
else:
    print("Query will not use indexes for optimization")

Index Scan on column 'embedding'
├── Sort by: COSINE_SIMILARITY(embedding, ARRAY[0.332268, 0.316658, 0.661787, 0.363573, 0.293287, 0.494711, 0.035018, 0.983019, 0.344806, 0.395443, 0.224208, 0.922944, 0.917482, 0.000156, 0.940689, 0.376213, 0.590671, 0.510595, 0.038578, 0.757781, 0.914312, 0.349699, 0.170851, 0.113930]) DESC
└── Limit (rows=3)
Execution plan: {'Plan': "Index Scan on column 'embedding'", 'children': [{'Plan': 'Sort by: COSINE_SIMILARITY(embedding, ARRAY[0.332268, 0.316658, 0.661787, 0.363573, 0.293287, 0.494711, 0.035018, 0.983019, 0.344806, 0.395443, 0.224208, 0.922944, 0.917482, 0.000156, 0.940689, 0.376213, 0.590671, 0.510595, 0.038578, 0.757781, 0.914312, 0.349699, 0.170851, 0.113930]) DESC'}, {'Plan': 'Limit (rows=3)'}]}
Query will not use indexes for optimization


In [182]:
explain_dict.keys()

dict_keys(['Plan', 'children'])

## 4*. Example Application

**Optional**

Prepare an example application that uses multimodal data. You can use Gradio
(https://www.gradio.app/guides/quickstart) or Streamlit (https://docs.streamlit.io/) or any other
framework of your choice.

The application should support the following features:
- dataset browsing: allow users to view the dataset by displaying objects consisting of two
  modalities (images and texts) in a user-friendly format, for example as a paginated table
  containing image and text columns. You may consider scaling images to a fixed width to avoid
  UI issues.
- search functionality: allow users to perform searches using the functions developed in the
  previous section, based on either a text query entered in a text field or an image uploaded by the
  user.